# Home Credit — ML / DSS
**Data Warehouse & DSS** · Notebook chạy trên Google Colab

Notebook này: đọc `application_train.csv` + `previous_application.csv` từ Google Drive → dựng **đúng feature contract v1** của người 1 →
chia train/valid/holdout theo `application_id` → baseline + Logistic Regression → đánh giá holdout → policy DSS + what-if →
lưu artifact, manifest, metrics, ví dụ response để **bàn giao cho FE**.

**Quy tắc đã khóa trong code:** TARGET không bao giờ nằm trong X · imputer/scaler/encoder chỉ `fit` trên train · chọn C và ngưỡng trên valid ·
holdout chỉ dùng để báo cáo · không số liệu giả: mọi con số trong báo cáo sinh ra từ lần chạy này.

> Chạy tuần tự **Runtime → Run all**. Chỉ cần sửa ô cấu hình ở mục 1 nếu thư mục Drive khác mặc định.

## 1. Cấu hình và kết nối Google Drive

In [ ]:
import os, sys, glob, json, time, hashlib, datetime as dt
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ====== SỬA Ở ĐÂY nếu cần ======
PROJECT_DIR = Path(os.environ.get("CREDIT_DW_ROOT", "/content/drive/MyDrive/credit_dw"))
CODE_DIR    = Path(os.environ.get("CREDIT_ML_CODE_DIR", PROJECT_DIR / "ml"))        # thư mục chứa credit_ml/
DATA_DIR    = Path(os.environ.get("CREDIT_DW_DATA_DIR", PROJECT_DIR / "data"))      # chứa 2 file CSV
OUT_DIR     = Path(os.environ.get("CREDIT_ML_OUT_DIR", PROJECT_DIR / "ml_outputs")) # nơi ghi kết quả
SAVE_SNAPSHOT = True                        # lưu snapshot feature+nhãn (csv.gz) để tái lập
# =================================

def find_file(name):
    p = DATA_DIR / name
    if p.exists():
        return p
    root = Path("/content/drive/MyDrive")
    hits = [Path(h) for h in glob.glob(str(root / "**" / name), recursive=True)] if root.exists() else []
    if len(hits) == 1:
        print(f"Không thấy {p}; dùng file tìm được: {hits[0]}")
        return hits[0]
    raise FileNotFoundError(f"Không tìm thấy {name} trong {DATA_DIR}. Tìm được {len(hits)} bản: {hits}. Sửa DATA_DIR.")

APP_CSV, PREV_CSV = find_file("application_train.csv"), find_file("previous_application.csv")
sys.path.insert(0, str(CODE_DIR))
import numpy as np, pandas as pd, sklearn, matplotlib
import matplotlib.pyplot as plt
import credit_ml
from credit_ml import config as C, features as F, modeling as M, training as T, artifacts as A, inference as I, whatif as W
from credit_ml.policy import PolicyConfig
OUT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR := OUT_DIR / "reports").mkdir(exist_ok=True)
(HANDOFF_DIR := OUT_DIR / "handoff").mkdir(exist_ok=True)
print("credit_ml", credit_ml.__version__, "| sklearn", sklearn.__version__, "| pandas", pd.__version__)
print("APP :", APP_CSV, f"({APP_CSV.stat().st_size/1e6:.0f} MB)")
print("PREV:", PREV_CSV, f"({PREV_CSV.stat().st_size/1e6:.0f} MB)")
print("OUT :", OUT_DIR)

## 2. Đọc dữ liệu và dựng feature contract v1
Dùng **đúng luật làm sạch của `sql/003_load_dw.sql` + `sql/002_views.sql`**: tuổi từ `DAYS_BIRTH`, `DAYS_EMPLOYED > 0` (kể cả 365243) → NULL,
tiền âm / `EXT_SOURCE` ngoài [0,1] → NULL, nhãn rỗng → `Unknown`, lịch sử chỉ tính `DAYS_DECISION <= 0` và **gộp về 1 dòng/hồ sơ trước khi ghép** (không nhân dòng).

In [ ]:
t0 = time.time()
app  = F.read_application_csv(APP_CSV)
prev = F.read_previous_csv(PREV_CSV)
X_all, labels, build_report = F.build_feature_table(app, prev)
y_all = labels["target"].astype(int)
assert X_all["application_id"].is_unique and len(X_all) == len(labels) == len(app)
assert "target" not in X_all.columns
print(f"{len(X_all):,} hồ sơ | {len(prev):,} hồ sơ trước | {time.time()-t0:.0f}s")
print("Báo cáo build:", json.dumps(build_report, indent=1))
print(f"Tỷ lệ TARGET=1 (khó khăn thanh toán ghi nhận): {y_all.mean():.4%}  ({int(y_all.sum()):,}/{len(y_all):,})")
del app, prev

In [ ]:
miss = X_all[C.PREDICTORS].isna().mean().sort_values(ascending=False).rename("missing_share").to_frame()
display(miss[miss.missing_share > 0].style.format("{:.2%}"))
display(X_all[C.NUMERIC_FEATURES].describe().T[["count","mean","std","min","50%","max"]])

## 3. Chia train / valid / holdout
Phân tầng theo TARGET, **chia theo `application_id`**, ba tập không giao nhau (60/20/20, `seed=42`). Danh sách ID được lưu để tái lập (seed không thay được danh sách ID).
Train: fit pipeline. Valid: chọn `C`, chọn ngưỡng phân loại, xem lại ngưỡng policy. Holdout: chỉ báo cáo.

In [ ]:
split = M.make_split(X_all["application_id"], y_all)
ids = {s: set(split.loc[split.split == s, "application_id"]) for s in ("train", "valid", "holdout")}
assert not (ids["train"] & ids["valid"]) and not (ids["train"] & ids["holdout"]) and not (ids["valid"] & ids["holdout"])
m_tr, m_va, m_ho = [(split["split"] == s).values for s in ("train", "valid", "holdout")]
X_tr, X_va, X_ho = X_all[m_tr], X_all[m_va], X_all[m_ho]
y_tr, y_va, y_ho = y_all[m_tr], y_all[m_va], y_all[m_ho]
(OUT_DIR / "splits").mkdir(exist_ok=True)
split.to_csv(OUT_DIR / "splits" / "split_ids.csv", index=False)
split_counts = {s: {"n": int(m.sum()), "positive_rate": float(y_all[m].mean())} for s, m in zip(("train","valid","holdout"), (m_tr,m_va,m_ho))}
print(json.dumps(split_counts, indent=1))

## 4. Chọn siêu tham số trên **valid**
Logistic Regression **không dùng `class_weight`** để điểm giữ ý nghĩa xác suất ước lượng (không bị thổi phồng). Pipeline: log1p cho cột tiền → impute trung vị + cờ thiếu → chuẩn hoá;
biến phân loại → `Unknown` + one-hot. Tất cả `fit` chỉ trên train.

In [ ]:
rows = []
for c in [0.001, 0.01, 0.1, 1.0, 10.0]:
    p = M.fit_pipeline(X_tr, y_tr, c)
    m = M.score_metrics(y_va, M.predict_score(p, X_va))
    rows.append({"C": c, "valid_roc_auc": m["roc_auc"], "valid_pr_auc": m["pr_auc"], "valid_log_loss": m["log_loss"]})
tuning = pd.DataFrame(rows)
display(tuning)
SELECTED_C = float(tuning.sort_values("valid_roc_auc", ascending=False).iloc[0]["C"])
print("C được chọn (valid ROC-AUC cao nhất):", SELECTED_C)

## 5. Huấn luyện mô hình cuối (chỉ trên train) và chuẩn bị policy

In [ ]:
pipe = M.fit_pipeline(X_tr, y_tr, SELECTED_C)
p_va = M.predict_score(pipe, X_va)
CLS_THRESHOLD = M.best_f1_threshold(y_va, p_va)          # chọn trên VALID
ref_stats = T.compute_reference_stats(pipe, X_tr)          # thống kê tham chiếu từ TRAIN
policy = PolicyConfig(ratio_extreme_threshold=ref_stats["credit_income_ratio_p99"])
print("Ngưỡng phân loại (F1 tối đa trên valid):", round(CLS_THRESHOLD, 4))
print("Policy:", json.dumps(policy.to_dict()))
print("\nPhân bố band trên VALID theo ngưỡng minh hoạ — để team xem có hợp lý không trước khi chốt:")
display(M.band_table(y_va, p_va, policy.review_threshold, policy.decline_threshold).style.format({"observed_rate": "{:.2%}", "share": "{:.2%}"}))

## 6. Đánh giá trên **holdout** (một lần)
Báo cáo song song hai baseline: (a) dự đoán hằng số theo tỷ lệ nhãn của train, (b) luật đơn giản dựa trung bình `EXT_SOURCE` có sẵn. Không dùng holdout để chọn gì.

In [ ]:
p_ho = M.predict_score(pipe, X_ho)
lr = M.score_metrics(y_ho, p_ho)
lr["roc_auc_ci95"] = M.bootstrap_auc_ci(y_ho, p_ho, n_boot=200)
prior = M.prior_baseline(y_tr)
p_prior = prior.predict_proba(np.zeros((len(y_ho), 1)))[:, 1]
base_prior = M.score_metrics(y_ho, p_prior)
base_rule = M.score_metrics(y_ho, M.ext_mean_rule_score(X_ho))
thr = {
    "f1_valid_threshold": M.threshold_metrics(y_ho, p_ho, CLS_THRESHOLD),
    "policy_review": M.threshold_metrics(y_ho, p_ho, policy.review_threshold),
    "policy_decline": M.threshold_metrics(y_ho, p_ho, policy.decline_threshold),
}
calib = M.calibration_table(y_ho, p_ho)
bands = M.band_table(y_ho, p_ho, policy.review_threshold, policy.decline_threshold)
cmp_ = pd.DataFrame({"Prior baseline": base_prior, "Luật EXT_SOURCE": base_rule, "Logistic Regression": lr}).loc[["n_evaluated","roc_auc","pr_auc","brier","log_loss"]]
display(cmp_)
print(f"ROC-AUC holdout = {lr['roc_auc']:.4f}  (CI 95% bootstrap: {lr['roc_auc_ci95'][0]:.4f} – {lr['roc_auc_ci95'][1]:.4f})")
print("\nConfusion matrix theo ngưỡng:")
display(pd.DataFrame(thr).T)
print("Hiệu chỉnh theo decile điểm (mean_score ≈ observed_rate nếu hiệu chỉnh tốt):")
display(calib.style.format({"mean_score": "{:.3f}", "observed_rate": "{:.3f}", "min_score": "{:.3f}", "max_score": "{:.3f}"}))
print("Band theo policy trên holdout:")
display(bands.style.format({"observed_rate": "{:.2%}", "share": "{:.2%}"}))

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
fig, ax = plt.subplots(1, 4, figsize=(20, 4.2))
for name, p in [("Logistic Regression", p_ho), ("EXT_SOURCE rule", M.ext_mean_rule_score(X_ho)), ("Prior", p_prior)]:
    fpr, tpr, _ = roc_curve(y_ho, p); ax[0].plot(fpr, tpr, label=name)
    pr, rc, _ = precision_recall_curve(y_ho, p); ax[1].plot(rc, pr, label=name)
ax[0].plot([0,1],[0,1],"k:",lw=.8); ax[0].set(title="ROC (holdout)", xlabel="FPR", ylabel="TPR"); ax[0].legend()
ax[1].set(title="Precision-Recall (holdout)", xlabel="Recall", ylabel="Precision"); ax[1].legend()
ax[2].plot(calib.mean_score, calib.observed_rate, "o-"); m = max(calib.mean_score.max(), calib.observed_rate.max())
ax[2].plot([0, m], [0, m], "k:"); ax[2].set(title="Calibration by decile (holdout)", xlabel="mean score", ylabel="observed TARGET=1 rate")
ax[3].hist(p_ho[y_ho.values == 0], bins=50, alpha=.6, density=True, label="TARGET=0")
ax[3].hist(p_ho[y_ho.values == 1], bins=50, alpha=.6, density=True, label="TARGET=1")
for t_, c_ in [(policy.review_threshold, "orange"), (policy.decline_threshold, "red")]: ax[3].axvline(t_, color=c_, ls="--")
ax[3].set(title="Score distribution (holdout)", xlabel="risk_score"); ax[3].legend()
plt.tight_layout(); plt.savefig(REPORT_DIR / "holdout_curves.png", dpi=130); plt.show()

coef = M.coefficient_table(pipe); coef.to_csv(REPORT_DIR / "coefficients.csv", index=False)
top = coef.head(15).iloc[::-1]
plt.figure(figsize=(7, 5)); plt.barh(top.feature, top.coef, color=["tab:red" if c > 0 else "tab:blue" for c in top.coef])
plt.title("Top 15 hệ số (chuẩn hoá) — đỏ: tăng rủi ro"); plt.tight_layout(); plt.savefig(REPORT_DIR / "top_coefficients.png", dpi=130); plt.show()

## 7. Lưu artifact + manifest, rồi **thử load lại** trước khi bàn giao
Manifest ghi: tên/phiên bản model, feature_version, sha256 artifact và 2 file CSV nguồn (khớp `ops.etl_runs.source_manifest` của DW), seed + danh sách ID chia tập, siêu tham số, metrics holdout, policy, phiên bản thư viện.

In [ ]:
today = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%d")
MODEL_REL_DIR = f"logreg_v1_{today}"
MODELS_ROOT = OUT_DIR / "models"
MODEL_DIR = MODELS_ROOT / MODEL_REL_DIR

if SAVE_SNAPSHOT:
    (OUT_DIR / "snapshots").mkdir(exist_ok=True)
    snap = X_all.merge(labels, on="application_id").merge(split, on="application_id")
    snap.to_csv(OUT_DIR / "snapshots" / "features_labels_v1.csv.gz", index=False)
    print("Snapshot:", (OUT_DIR / "snapshots" / "features_labels_v1.csv.gz").name)

pick = X_ho.iloc[0]
smoke_features = I.features_from_row(pick)
smoke_expected = float(M.predict_score(pipe, X_ho.iloc[[0]])[0])

evaluation_metrics = {
    "cohort": "application_train.csv — holdout stratified 20% (seed 42), chia theo application_id",
    "n_evaluated": lr["n_evaluated"], "n_positive": lr["n_positive"], "positive_rate": lr["positive_rate"],
    "roc_auc": lr["roc_auc"], "roc_auc_ci95_bootstrap": list(lr["roc_auc_ci95"]),
    "pr_auc": lr["pr_auc"], "brier": lr["brier"], "log_loss": lr["log_loss"],
    "threshold": CLS_THRESHOLD, "threshold_selected_on": "validation (F1 tối đa)",
    "confusion_matrix": {k: thr["f1_valid_threshold"][k] for k in ("tn", "fp", "fn", "tp")},
    "threshold_metrics": thr,
    "baselines": {"prior": {"roc_auc": base_prior["roc_auc"], "pr_auc": base_prior["pr_auc"]},
                  "ext_source_mean_rule": {"roc_auc": base_rule["roc_auc"], "pr_auc": base_rule["pr_auc"]}},
    "validation": M.score_metrics(y_va, p_va),
    "calibration_deciles": calib.to_dict("records"), "bands_holdout": bands.to_dict("records"),
    "definitions": "risk_score = P(TARGET=1) ước lượng; roc_auc = sklearn.roc_auc_score; pr_auc = average_precision_score; chưa kiểm chứng hiệu chỉnh như xác suất thực.",
}
manifest = T.build_manifest(
    model_name=C.MODEL_NAME, selected_C=SELECTED_C, tuning=tuning.to_dict("records"),
    split_counts=split_counts, split_ids_rel="../../splits/split_ids.csv",
    split_ids_sha=A.sha256_file(OUT_DIR / "splits" / "split_ids.csv"),
    data_files=[T.file_record(APP_CSV, "application_train", len(X_all)), T.file_record(PREV_CSV, "previous_application")],
    build_report=build_report,
    label_stats={"n_labeled": int(len(y_all)), "n_positive": int(y_all.sum()), "positive_rate": float(y_all.mean())},
    evaluation_metrics=evaluation_metrics, policy=policy, reference_stats=ref_stats,
    smoke_test={"application_id": int(pick["application_id"]), "features": smoke_features, "expected_risk_score": smoke_expected},
    notes=["Điểm rủi ro cao hơn = rủi ro ước lượng cao hơn (ngược scorecard 0-1000).",
           "Ngưỡng policy 0.10/0.25 là minh hoạ học thuật, chưa phải chính sách ngân hàng."])
manifest = A.save_bundle(MODEL_DIR, pipe, manifest)

# ---- Thử load lại như người 3 sẽ làm; so với pipeline trong bộ nhớ
bundle = I.load_bundle(MODELS_ROOT, MODEL_REL_DIR)
sample = X_ho.sample(min(2000, len(X_ho)), random_state=0)
via_api = np.array([I.assess(bundle, int(r["application_id"]), I.features_from_row(r))["risk_score"] for _, r in sample.iterrows()])
assert np.allclose(via_api, M.predict_score(pipe, sample), atol=1e-12), "assess() lệch pipeline"
assert np.isfinite(via_api).all() and ((via_api >= 0) & (via_api <= 1)).all()
print("Đã lưu:", MODEL_DIR)
print("sha256 artifact:", manifest["artifact"]["sha256"])
print("Load lại + chấm", len(sample), "hồ sơ holdout qua assess(): khớp pipeline gốc ✔")

## 8. Ví dụ response bàn giao (sinh từ model thật)
Một hồ sơ **holdout thật** (kết quả thành công), một ca cảnh báo thiếu dữ liệu (cùng hồ sơ nhưng gán `amt_income_total=None` — *chỉ để minh hoạ policy*), what-if, và các response lỗi `failed` / `not_ready` (lỗi mô tả cấu trúc, không có điểm).

In [ ]:
real_id = int(pick["application_id"])
ok = I.assess(bundle, real_id, smoke_features, model_run_id=None)
alert_features = {**smoke_features, "amt_income_total": None, "credit_income_ratio": None}
alert = I.assess(bundle, real_id, alert_features)
wi = W.what_if(bundle, real_id, smoke_features, income=(smoke_features["amt_income_total"] or 100000) * 1.5, review_threshold=0.08)
failed = I.failed_result(real_id, 1, ["INVALID_FEATURES:FORBIDDEN_KEY:target"], smoke_features)
not_ready = I.not_ready_result(real_id, "MODEL_NOT_READY")
examples = {
 "assessment_success_REAL_HOLDOUT_APPLICATION.json": ok,
 "assessment_data_alert_ILLUSTRATIVE_income_removed.json": alert,
 "what_if_ILLUSTRATIVE.json": wi,
 "assessment_failed_STRUCTURE_ONLY.json": failed,
 "assessment_not_ready_STRUCTURE_ONLY.json": not_ready,
}
for name, obj in examples.items():
    (HANDOFF_DIR / name).write_text(json.dumps(A.json_safe(obj), indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
print(json.dumps({k: ok[k] for k in ("status","risk_score","risk_band","recommendation","reason_codes")}, indent=1))
print("\nAlert case:", alert["recommendation"], alert["reason_codes"][:4])
print("What-if:", wi["original"]["recommendation"], "→", wi["scenario"]["recommendation"], "| Δscore =", round(wi["score_delta"], 5))

## 9. Sinh `HANDOFF.md` (số liệu thật từ lần chạy này) và liệt kê file bàn giao

In [ ]:
lib = A.library_versions()
tn, fp, fn, tp = (evaluation_metrics["confusion_matrix"][k] for k in ("tn", "fp", "fn", "tp"))
handoff = f"""# Bàn giao ML/DSS — người 2 → người 3 (và người 1)
Sinh tự động lúc {manifest['trained_at']} từ notebook. Mọi số liệu dưới đây lấy từ lần chạy này trên hai file CSV được chỉ định ở mục 1 (kiểm tra sha256 bên dưới).

## Model
- `{MODEL_REL_DIR}/` trong `ml_outputs/models/` — `pipeline.joblib` + `model_manifest.json`
- model_name `{manifest['model_name']}`, feature_version `{manifest['feature_version']}`, C = {SELECTED_C}
- sha256 artifact: `{manifest['artifact']['sha256']}`
- Môi trường train: Python {lib['python']}, scikit-learn {lib['scikit-learn']}, pandas {lib['pandas']}, numpy {lib['numpy']}, joblib {lib['joblib']}
  → khi chạy trên Windows: `pip install scikit-learn=={lib['scikit-learn']} joblib=={lib['joblib']}`

## Kết quả holdout ({evaluation_metrics['n_evaluated']:,} hồ sơ, {evaluation_metrics['n_positive']:,} TARGET=1)
| | ROC-AUC | PR-AUC |
|---|---|---|
| Prior baseline | {base_prior['roc_auc']:.4f} | {base_prior['pr_auc']:.4f} |
| Luật EXT_SOURCE | {base_rule['roc_auc']:.4f} | {base_rule['pr_auc']:.4f} |
| **Logistic Regression** | **{lr['roc_auc']:.4f}** (CI95 {lr['roc_auc_ci95'][0]:.4f}–{lr['roc_auc_ci95'][1]:.4f}) | **{lr['pr_auc']:.4f}** |

Ngưỡng phân loại {CLS_THRESHOLD:.4f} (chọn trên valid): TN={tn:,} FP={fp:,} FN={fn:,} TP={tp:,}.
Brier = {lr['brier']:.4f}; log-loss = {lr['log_loss']:.4f}. Xem bảng hiệu chỉnh trong `reports/` và `model_manifest.json` — chưa nên gọi risk_score là xác suất đã hiệu chỉnh nếu decile lệch.

## Policy (minh hoạ, chờ team duyệt) — `{policy.version}`
- Thiếu dữ liệu quan trọng / cảnh báo mức cao → `REVIEW`; nếu không: p ≥ {policy.decline_threshold} → `DECLINE_SIMULATION`; {policy.review_threshold} ≤ p < {policy.decline_threshold} → `REVIEW`; p < {policy.review_threshold} → `APPROVE_SIMULATION`.
- risk_band: LOW / MEDIUM / HIGH theo cùng hai ngưỡng.

## Dữ liệu nguồn (sha256 phải khớp ops.etl_runs.source_manifest; register_model.py kiểm tra)
""" + "\n".join(f"- {f['role']}: `{f['file']}` sha256 `{f['sha256']}`" for f in manifest["data"]["files"]) + """

## Hạn chế
Nhãn TARGET là định nghĩa khó khăn thanh toán của nguồn, không phải vỡ nợ 90 ngày/12 tháng. Recommendation chỉ là mô phỏng, không phải quyết định cấp tín dụng. Số lịch sử previous chỉ mô tả dữ liệu đã nạp.
"""
(HANDOFF_DIR / "HANDOFF.md").write_text(handoff, encoding="utf-8")
(REPORT_DIR / "metrics.json").write_text(json.dumps(A.json_safe(evaluation_metrics), indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
print(handoff)
print("\n=== Các file đã tạo trong", OUT_DIR, "===")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file(): print(f"{p.relative_to(OUT_DIR)}  ({p.stat().st_size/1e3:,.0f} KB)")